# Practice

In [18]:
from cs336_basics.utils.tokenization_utils import load_vocab_and_merges, save_vocab_and_merges
vocab, merges = load_vocab_and_merges(vocab_path= "checkpoints/owt_vocab.json", merges_path= "checkpoints/owt_merges.txt")

In [9]:
a = [1,3,5,7,9]
t = yield from a
for x in t:
    print(x)


SyntaxError: 'yield from' outside function (1691946854.py, line 2)

In [1]:
import regex as re
texts = "This is a test string<|endoftext|> This is a <unk><|endoftext|> new text.<|endoftext|>"
special_tokens = ["<unk>", "<|endoftext|>"]
special_tokens_pattern = "|".join(re.escape(tok) for tok in special_tokens)
pretokenization_pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
for match in re.finditer(f"{special_tokens_pattern}|{pretokenization_pattern}", texts):
    print(match.start(), match.end(), match.group(0))

0 4 This
4 7  is
7 9  a
9 14  test
14 21  string
21 34 <|endoftext|>
34 39  This
39 42  is
42 44  a
44 46  <
46 49 unk
49 52 ><|
52 61 endoftext
61 63 |>
63 67  new
67 72  text
72 75 .<|
75 84 endoftext
84 86 |>


In [2]:
pretokenization_pattern

"'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+"

In [3]:
from cs336_basics.utils.tokenization_utils import find_chunk_boundaries_for_encoding
with open("data/TinyStoriesV2-GPT4-train-10k.txt", "r", encoding="utf-8") as f:
    text = f.read(1024)
    boundaries = find_chunk_boundaries_for_encoding(f, pretokenization_pattern, special_tokens=special_tokens, chunk_size=1024, mini_chunk_size=1024)
boundaries[:10]

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x9c in position 0: invalid start byte

In [32]:
from tests.common import FIXTURES_PATH, gpt2_bytes_to_unicode
from tests.test_tokenizer import get_tokenizer_from_vocab_merges_path, _encode_iterable

VOCAB_PATH = FIXTURES_PATH / "gpt2_vocab.json"
MERGES_PATH = FIXTURES_PATH / "gpt2_merges.txt"

tokenizer = get_tokenizer_from_vocab_merges_path(
    vocab_path=VOCAB_PATH,
    merges_path=MERGES_PATH,
)
max = 0
i = 0
with open(FIXTURES_PATH / "tinystories_sample_5M.txt") as f:
    for text in f:
        if len(text) > max:
            max = len(text)
        # if i > 50:
        #     break
        # i += 1
print(max)

3154


In [27]:
!pytest tests/test_tokenizer.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.0.2, pluggy-1.6.0
rootdir: /home/po/Projects/cs336/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 25 items                                                             

tests/test_tokenizer.py::test_roundtrip_empty PASSED
tests/test_tokenizer.py::test_empty_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_single_character PASSED
tests/test_tokenizer.py::test_single_character_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_single_unicode_character PASSED
tests/test_tokenizer.py::test_single_unicode_character_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_ascii_string PASSED
tests/test_tokenizer.py::test_ascii_string_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_unicode_string PASSED
tests/test_tokenizer.py::test_unicode_string_matches_tiktoken PASSED
t

In [5]:
from cs336_basics.bpe_tokenizer import BPETokenizer
tkz = BPETokenizer.from_file(vocab_filepath= "checkpoints/owt_vocab.json", merges_filepath= "checkpoints/owt_merges.txt", special_tokens= ["<|endoftext|>"], pretokenization_pattern=pretokenization_pattern)

In [21]:
i=0
j = 0
with open("data/TinyStoriesV2-GPT4-train-10k.txt", "r", encoding="utf-8") as f:
    for x in f:
        print(x)
        print(tkz.encode(x))
        j+=1
        if j>3:
            break
    f.seek(0)
    for token in tkz.encode_iterable(f):
        print(token, tkz.vocab[token])
        i += 1
        if i > 10:
            break



[10]
Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  

[7222, 3059, 258, 630, 592, 359, 258, 1310, 2905, 3313, 3676, 46, 3676, 5812, 284, 6800, 262, 918, 1026, 693, 46, 687, 2351, 810, 4996, 1188, 44, 561, 5214, 410, 1433, 317, 520, 318, 3756, 287, 258, 2768, 46, 1916, 1084, 44, 3676, 359, 5921, 830, 262, 2768, 613, 341, 1510, 1779, 258, 844, 2048, 410, 623, 46, 1787, 3676, 2351, 340, 341, 359, 23267, 33, 32, 32, 10]
He said, “Wow, that is a really amazing vase! Can I buy it?” 

[1376, 480, 44, 451, 21167, 44, 317, 321, 258, 1043, 4996, 410, 623, 33, 1467, 316, 2770, 340, 2969, 32, 10]
The shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”

[445, 6031, 12878, 18657, 294, 480, 44, 4

b'So'

In [1]:
texts = "this <|endoftext|> This is a <unk><|endoftext|> new text.<|endoftext|>"
texts = "this is a text without special toke"
for match in re.finditer(pretokenization_pattern, texts):
    print(match.start(), match.end(), match.group(0))
#re.findall(pretokenization_pattern, texts)

NameError: name 're' is not defined

In [16]:
len(texts)

18

In [21]:
from cs336_basics.utils.tokenization_utils import get_chunk_boundaries

In [22]:
from importlib import reload
import cs336_basics.utils.tokenization_utils as utl
reload(utl)

<module 'cs336_basics.utils.tokenization_utils' from '/home/po/Projects/cs336/assignment1-basics/cs336_basics/utils/tokenization_utils.py'>

In [3]:
chunk_size = None
num_chunks = 5
path = "data/TinyStoriesV2-GPT4-valid.txt"
#path = "data/TinyStoriesV2-GPT4-train-10k.txt"

In [19]:
from typing import BinaryIO
with open(path, "rb") as file:
    if isinstance(file, BinaryIO):
        print("file is a BinaryIO")
    
    boundaries =utl.get_chunk_boundaries(file, split_token=b"<|endoftext|>", chunk_size=chunk_size, num_chunks=num_chunks)
boundaries

<_io.BufferedReader name='data/TinyStoriesV2-GPT4-valid.txt'> b'<|endoftext|>' 5 None


[(0, 4500912),
 (4500912, 9001679),
 (9001679, 13502026),
 (13502026, 18002387),
 (18002387, 22502601)]

In [ ]:
# chunk_size = None
# num_chunks = 2
# #path = "data/TinyStoriesV2-GPT4-valid.txt"
# path = "data/TinyStoriesV2-GPT4-train-10k.txt"
with open(path, "r") as f:
    texts = f.read()
    print(len(texts))
    print(utl.get_chunk_boundaries(texts, split_token="<|endoftext|>", chunk_size=chunk_size, num_chunks=num_chunks))
with open(path, "rb") as f:
    print(utl.get_chunk_boundaries(file=f, split_token=b"<|endoftext|>", chunk_size=chunk_size, num_chunks=num_chunks))

22493387
[(0, 242), (242, 4499056), (4499056, 8997918), (8997918, 13496519), (13496519, 17994987)]


ValueError: file type: <class '_io.TextIOWrapper'>; file must be either a BinaryIO or a string

In [4]:
texts

'This is a test string<|endoftext|> This is a <unk><|endoftext|> new text.<|endoftext|>'

In [11]:
import regex as re
texts = "This is a test string<|endoftext|><|endoftext|><|endoftext|> This is a <unk><|endoftext|> new text.<|endoftext|>"
#special_tokens = ["<unk>", "<|endoftext|><|endoftext|>", "<|endoftext|>"]
special_tokens = ["<unk>", "<|endoftext|>", "<|endoftext|><|endoftext|>"]
#special_tokens.sort(reverse=True)
print(special_tokens)
pattern = re.compile("(" + "|".join(re.escape(tok) for tok in sorted(special_tokens, reverse=True)) + ")" if special_tokens else None)
pattern.split(texts)

['<unk>', '<|endoftext|>', '<|endoftext|><|endoftext|>']


['This is a test string',
 '<|endoftext|><|endoftext|>',
 '',
 '<|endoftext|>',
 ' This is a ',
 '<unk>',
 '',
 '<|endoftext|>',
 ' new text.',
 '<|endoftext|>',
 '']

In [13]:
pretokenization_pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
ids = []
print(texts)
if special_tokens is not None:
    special_tokens_pattern = "("+ "|".join(re.escape(token) for token in sorted(special_tokens, reverse=True)) + ")"
for text in re.split(special_tokens_pattern, texts):
    if text=="":
        continue
    elif text in special_tokens:
        ids.append(text)
    else:
        words = re.findall(pretokenization_pattern, text)
        ids.extend(words)
ids

This is a test string<|endoftext|><|endoftext|><|endoftext|> This is a <unk><|endoftext|> new text.<|endoftext|>


['This',
 ' is',
 ' a',
 ' test',
 ' string',
 '<|endoftext|><|endoftext|>',
 '<|endoftext|>',
 ' This',
 ' is',
 ' a',
 ' ',
 '<unk>',
 '<|endoftext|>',
 ' new',
 ' text',
 '.',
 '<|endoftext|>']

In [17]:
f"{sorted(special_tokens, reverse=True)}|{pretokenization_pattern}"

"['<|endoftext|><|endoftext|>', '<|endoftext|>', '<unk>']|'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+"

In [27]:
special_pattern = "|".join(re.escape(t) for t in sorted(special_tokens, reverse=True))
combined_pattern = f"({special_pattern})|{pretokenization_pattern}"
for match in re.finditer(combined_pattern, texts):
    print(match.group(0))
    #print(match)

This
 is
 a
 test
 string
<|endoftext|><|endoftext|>
<|endoftext|>
 This
 is
 a
 <
unk
><|
endoftext
|>
 new
 text
.<|
endoftext
|>


In [20]:
special_tokens_pattern = "|".join(re.escape(token) for token in sorted(special_tokens, reverse=True))
re.findall(f"{special_tokens_pattern}|{pretokenization_pattern}", texts)

['This',
 ' is',
 ' a',
 ' test',
 ' string',
 '<|endoftext|><|endoftext|>',
 '<|endoftext|>',
 ' This',
 ' is',
 ' a',
 ' <',
 'unk',
 '><|',
 'endoftext',
 '|>',
 ' new',
 ' text',
 '.<|',
 'endoftext',
 '|>']

In [17]:
texts_bytes = texts.encode("utf-8")
tuple(texts_bytes)

(84,
 104,
 105,
 115,
 32,
 105,
 115,
 32,
 97,
 32,
 116,
 101,
 115,
 116,
 32,
 115,
 116,
 114,
 105,
 110,
 103,
 60,
 124,
 101,
 110,
 100,
 111,
 102,
 116,
 101,
 120,
 116,
 124,
 62,
 32,
 84,
 104,
 105,
 115,
 32,
 105,
 115,
 32,
 97,
 32,
 60,
 117,
 110,
 107,
 62,
 60,
 124,
 101,
 110,
 100,
 111,
 102,
 116,
 101,
 120,
 116,
 124,
 62,
 32,
 110,
 101,
 119,
 32,
 116,
 101,
 120,
 116,
 46,
 60,
 124,
 101,
 110,
 100,
 111,
 102,
 116,
 101,
 120,
 116,
 124,
 62)

In [59]:
from importlib import reload
import cs336_basics.bpe_tokenizer
reload(cs336_basics.bpe_tokenizer)

<module 'cs336_basics.bpe_tokenizer' from '/home/po/Projects/cs336/assignment1-basics/cs336_basics/bpe_tokenizer.py'>

In [60]:
from cs336_basics.bpe_tokenizer import BPETokenizer
tokenizer = BPETokenizer.from_file(vocab_filepath = "checkpoints/owt_vocab.json", merges_filepath = "checkpoints/owt_merges.txt", special_tokens = ["<|endoftext|>"])
print(texts)
enc = tokenizer.encode(texts)
print(enc)

This is a test string<|endoftext|> This is a <unk><|endoftext|> new text.<|endoftext|>
[1202, 321, 258, 1352, 6295, 256, 851, 321, 258, 2668, 3119, 62, 256, 597, 2946, 46, 256]


In [61]:
texts == tokenizer.decode(enc)

True

In [56]:
file = open("data/TinyStoriesV2-GPT4-valid.txt", "r", encoding="utf-8")
i = 0
for x in tokenizer.encode_iterable(file):
    print(x)
    i += 1
    if i > 100:
        break

117
795
39
116
412
284
309
11011
288
262
7736
3565
44
316
39
299
1780
361
2780
393
9190
2863
521
3356
352
262
1310
2767
46
1439
359
844
1517
294
262
9190
2435
1510
284
3650
633
46
687
25818
929
633
294
678
3888
693
3356
46
393
9190
528
1021
459
1230
1527
46
10
256
10
7222
3059
258
630
44
287
258
6196
294
25105
1266
44
592
359
258
1109
5043
46
331
1310
2905
3313
3872
7961
284
665
1993
262
5043
46
1916
1084
44
3872
2390
459
2388
2332


# Answers

##### Problem (train_bpe_expts_owt): BPE Training on OpenWebText (2 points)

In [21]:
from cs336_basics.utils.tokenization_utils import load_vocab_and_merges, save_vocab_and_merges
vocab, merges = load_vocab_and_merges(vocab_path= "checkpoints/owt_vocab.json", merges_path= "checkpoints/owt_merges.txt")

In [24]:
# longest token in the vocab
longest_token = max(vocab.values(), key=len)
longest_token

b'\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82'

In [25]:
longest_token.decode('utf-8')

'ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ'

In [28]:
t = 'Ã'
len(t.encode('utf-8'))

2

In [30]:
longest = vocab[0]
ln = len(longest)
for i in range(32000):
    if len(vocab[i]) > len(longest):
        longest = vocab[i]
        if len(longest) >= 2*ln:
            print(i, longest.decode('utf-8'))
            ln = len(longest)

256 <|endoftext|>
10892 --------------------------------
25799 ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ


# Training

In [ ]:
!git clone -b dev https://github.com/soumitrapy/cs336_assignment1-basics.git
%cd cs336_assignment1-basics

In [ ]:
!mkdir data
%cd data
#!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
#!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
!gunzip owt_train.txt.gz
#!wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
#!gunzip owt_valid.txt.gz

%cd ..
!ls data/

In [ ]:
# %cd data
# !gunzip owt_train.txt.gz
# %cd ..

In [ ]:
#!uv run --with wrapt python -m cs336_basics.bpe_training --input_path="data/TinyStoriesV2-GPT4-train.txt" --num_chunks=4 --vocab_size=10000 --vocab_path="checkpoints/tinystories_vocab.json" --merges_path="checkpoints/tinystories_merges.txt"
!uv run --with wrapt python -m cs336_basics.bpe_training --input_path="data/owt_train.txt" --num_chunks=4 --vocab_size=32000 --vocab_path="checkpoints/owt_vocab.json" --merges_path="checkpoints/owt_merges.txt"

In [ ]:
import zipfile
import base64
import os
from IPython.display import HTML, display

filename = "owt"
#filename = "tinystories"
zip_name = f'{filename}.zip'


# 1. Create the zip file
with zipfile.ZipFile(f'/kaggle/working/{zip_name}', 'w') as zf:
    zf.write(f'/kaggle/working/cs336_assignment1-basics/checkpoints/{filename}_vocab.json', arcname=f'{filename}_vocab.json')
    zf.write(f'/kaggle/working/cs336_assignment1-basics/checkpoints/{filename}_merges.txt', arcname=f'{filename}_merges.txt')

# 2. Generate download link for the zip
with open(f'/kaggle/working/{zip_name}', "rb") as f:
    data = f.read()
    
b64 = base64.b64encode(data).decode()
payload = f"data:application/octet-stream;base64,{b64}"

display(HTML(f'<a download="{zip_name}" href="{payload}" target="_blank">Download All ({zip_name})</a>'))